<a href="https://colab.research.google.com/github/juliapereira-create/TCC---2026/blob/V1/TCC_Batista_e_Clabunde_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/juliapereira-create/TCC---2026/blob/main/TCC_Batista_e_Clabunde_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🏥 TCC — Otimização da Programação de Manutenção de Equipamentos Médicos
via Pesquisa Operacional (PLIM) e AHP.

Welton e Julia - Versão ???
--
### 🎯 Nosso objetivo
Minimizar os impactos da manutenção preventiva sobre a agenda clínica,
evitando cancelamentos de exames e garantindo a disponibilidade de técnicos
para acompanhamento — usando **Programação Linear Inteira Mista (PLIM) e AHP**.


---
## 📦 Bloco 1 — Bibliotecas



In [10]:
# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Instalação das bibliotecas
# PuLP: resolve o modelo de otimização (PLIM)
# Pandas: organiza os dados em tabelas
# Matplotlib + Seaborn: gera os gráficos e o heatmap  que o professor pediu
# ─────────────────────────────────────────────────────────────

!pip install -q --upgrade pulp pandas matplotlib seaborn numpy

import pulp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')


---
## 🔧 Bloco 2 — Dados dos Equipamentos

### ⏱️ Tempo de manutenção preventiva
Valores compatíveis com contratos de engenharia clínica no Brasil e manuais de fabricantes:
- **Tomógrafo (8h)** e **Ressonância (10h)**: confirmados pela prática de mercado (Siemens, GE, Philips).
  - Ref: [VX Med — Manutenção de Equipamentos Radiológicos](https://vx.med.br/manutencao-de-equipamentos-radiologicos/)
- **Ultrassom (3h)**: POP-STEC-079 do HU-UFGD / Portaria nº 094/2025.
  - Ref: [gov.br — POP Ultrassonografia](https://www.gov.br/hubrasil/pt-br/hospitais-universitarios/regiao-centro-oeste/hu-ufgd/acesso-a-informacao/pops-protocolos-e-processos/gad/pop-stec-079-manutencao-preventiva-de-equipamentos-do-tipo-sistema-de-ultrassonografia.pdf)
- **Mamógrafo (4h)** e **Raio-X Digital (4h)**: Portaria SVS/MS nº 453/1998 e RDC 611/2022.
  - Ref: [Portaria 453 — Legisweb](https://www.legisweb.com.br/legislacao/?id=181745)
- **Hemodinâmica (8h)**: alta complexidade, procedimento em sala dedicada — RDC 611/2022.
  - Ref: [RDC 611/2022 — Sapra Landauer](https://www.sapralandauer.com.br/wp-content/uploads/2022/05/RESOLU%C3%87%C3%83O-RDC-N%C2%BA-611-DE-9-DE-Mar%C3%A7o-DE-2022.pdf)
- **Arco Cirúrgico (5h)**: equipamento de fluoroscopia intraoperatória — valores praticados em EC hospitalar.

### ⏱️ Tempo médio de exame
- **Tomógrafo (30 min)**: Hospital São José — "duração varia entre 15 e 30 min".
  - Ref: [Hospital São José — Tomografia](https://www.hsjose.com.br/exames/centro-diagnosticos/tomografia)
- **Ressonância (60 min)**: exame mais longo por natureza do campo magnético.
  - Ref: [Blume Diagnósticos — Exames de Imagem](https://blumediagnosticos.com.br/exames-de-imagem-quando-fazer-ressonancia-magnetica-tomografia-e-ultrassonografia/)
- **Mamógrafo (20 min)**, **Ultrassom (25 min)**, **Raio-X (15 min)**: valores práticos amplamente aceitos na literatura clínica brasileira.
- **Arco Cirúrgico (90 min)** e **Hemodinâmica (120 min)**: procedimentos cirúrgicos/intervencionistas com tempo mínimo de sala.

### 📅 Periodicidade de manutenção preventiva
De acordo com a **RDC 16/2013 (Anvisa)** e **Portaria SVS/MS nº 453/1998**:
- Equipamentos de imagem ionizante (TC, RM, Raio-X, Mamógrafo, Hemodinâmica, Arco): manutenção a cada **30 dias** (controle de qualidade mensal) e revisão completa a cada **180 dias** (semestral).
- **Hemodinâmica**: pelo alto uso e criticidade, periodicidade mais curta — **30 e 90 dias**.
- Ref: [RDC 16/2013 — BVS Saúde](https://bvsms.saude.gov.br/bvs/saudelegis/anvisa/2013/rdc0016_28_03_2013.pdf)


In [11]:
# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Dados dos equipamentos
#
# Cada equipamento tem:
#   - qtd          : quantidade de unidades no hospital
#   - tem_backup   : True se existe equipamento reserva disponível
#   - tempo_mp_h   : horas de manutenção preventiva (por revisão)
#   - tempo_exame  : duração média de um exame em minutos
#   - period_dias  : lista com as periodicidades de MP em dias
#                    [30, 180] = mensal leve + semestral completa
#   - tipo_mp      : 'externa' (terceirizada) ou 'interna' (equipe do hospital)
# ─────────────────────────────────────────────────────────────

equipamentos = {

    # ── Tomógrafo Computadorizado ──────────────────────────
    # Equipamento de alto custo e alta criticidade.
    # Sem backup — qualquer manutenção exige planejamento cuidadoso.
    # MP exige equipe especializada do fabricante (OEM).
    'Tomografo': {
        'nome':         'Tomógrafo Computadorizado',
        'qtd':          1,
        'tem_backup':   False,
        'tempo_mp_h':   8,      # horas — ref: prática de mercado OEM (Siemens/GE/Philips)
        'tempo_exame':  30,     # minutos — ref: hsjose.com.br (15–30 min)
        'period_dias':  [30, 180],  # mensal + semestral (RDC 16/2013 + Port. 453/98)
        'tipo_mp':      'externa',
    },

    # ── Ressonância Magnética ──────────────────────────────
    # Equipamento de maior complexidade e custo do parque.
    # Sem backup — paralisação impacta diretamente o diagnóstico.
    # MP envolve campo magnético, gases criogênicos (hélio) e software.
    'Ressonancia': {
        'nome':         'Ressonância Magnética',
        'qtd':          1,
        'tem_backup':   False,
        'tempo_mp_h':   10,     # horas — maior complexidade, inclui testes de phantom
        'tempo_exame':  60,     # minutos — ref: blumediagnosticos.com.br
        'period_dias':  [30, 180],
        'tipo_mp':      'externa',
    },

    # ── Mamógrafo Digital ──────────────────────────────────
    # Equipamento de rastreamento. Alto volume de exames.
    # Sem backup — paralisação gera fila de espera significativa.
    # Regulado pela Portaria 453/98 e RDC 611/2022.
    'Mamografo': {
        'nome':         'Mamógrafo Digital',
        'qtd':          1,
        'tem_backup':   False,
        'tempo_mp_h':   4,      # horas — ref: Portaria 453/98
        'tempo_exame':  20,     # minutos — ref: prática clínica brasileira
        'period_dias':  [30, 180],
        'tipo_mp':      'interna',
    },

    # ── Arco Cirúrgico (Arco em C) ─────────────────────────
    # Equipamento de fluoroscopia intraoperatória.
    # 2 unidades = backup disponível (se um for para MP, o outro assume).
    # Usado em Centro Cirúrgico — paralisação impacta cirurgias.
    'Arco_Cirurgico': {
        'nome':         'Arco Cirúrgico (Arco em C)',
        'qtd':          2,
        'tem_backup':   True,   # 2 unidades — uma faz backup da outra
        'tempo_mp_h':   5,      # horas — ref: contratos EC hospitalar
        'tempo_exame':  90,     # minutos — tempo mínimo de cirurgia
        'period_dias':  [30, 180],
        'tipo_mp':      'externa',
    },

    # ── Ultrassom ──────────────────────────────────────────
    # 6 aparelhos — volume altíssimo de exames.
    # Backup disponível pela quantidade de unidades.
    # Regulado pela IN 96 (Anvisa) + POP-STEC-079.
    'Ultrassom': {
        'nome':         'Aparelho de Ultrassom',
        'qtd':          6,
        'tem_backup':   True,   # 6 unidades — ampla capacidade de backup
        'tempo_mp_h':   3,      # horas — ref: POP-STEC-079/HU-UFGD (2025)
        'tempo_exame':  25,     # minutos — ref: prática clínica
        'period_dias':  [30, 180],
        'tipo_mp':      'interna',
    },

    # ── Raio-X Digital ─────────────────────────────────────
    # Alta demanda de exames. Única unidade.
    # Regulado pela Portaria 453/98 e RDC 611/2022.
    'RaioX': {
        'nome':         'Raio-X Digital',
        'qtd':          1,
        'tem_backup':   False,
        'tempo_mp_h':   4,      # horas — ref: Portaria 453/98
        'tempo_exame':  15,     # minutos — exame rápido por natureza
        'period_dias':  [30, 180],
        'tipo_mp':      'interna',
    },

    # ── Hemodinâmica ───────────────────────────────────────
    # Sala de hemodinâmica — procedimentos intervencionistas cardíacos.
    # 2 salas = backup disponível.
    # Alta criticidade clínica. Periodicidade mais curta (90 dias)
    # por exigência da RDC 611/2022 para equipamentos intervencionistas.
    'Hemodinamica': {
        'nome':         'Hemodinâmica',
        'qtd':          2,
        'tem_backup':   True,   # 2 salas — uma faz backup da outra
        'tempo_mp_h':   8,      # horas — alta complexidade (detectors, gantry, SW)
        'tempo_exame':  120,    # minutos — procedimento intervencionista
        'period_dias':  [30, 90],   # mais curto — exigência RDC 611/2022
        'tipo_mp':      'externa',
    },
}

print('Descrição das quantidades de equipamentos existentes')
print(f'Total de tipos de equipamentos: {len(equipamentos)}')
print(f'Total de unidades: {sum(v["qtd"] for v in equipamentos.values())}')


Descrição das quantidades de equipamentos existentes
Total de tipos de equipamentos: 7
Total de unidades: 14


---
## ⚖️ Bloco 3 — Pesos de Criticidade (Método AHP)

O **AHP (Analytic Hierarchy Process)** é um método de tomada de decisão multicritério
amplamente usado em engenharia clínica para priorizar equipamentos.
Referência: Masmoudi et al. (2016) — *Decision support procedure for Medical Equipment Maintenance Management*.

### Critérios usados para definir criticidade

| Critério | Peso |
|----------|------|
| Risco ao paciente em caso de falha | 40% |
| Custo de substituição / reposição | 25% |
| Volume de exames (impacto operacional) | 20% |
| Facilidade de backup | 15% |

### Ranking de criticidade resultante

1. 🔴 **Hemodinâmica** — risco de vida imediato em falha durante procedimento intervencionista
2. 🔴 **Ressonância Magnética** — alto custo, sem backup, impacto neurológico/oncológico
3. 🔴 **Tomógrafo** — diagnóstico crítico de urgência, sem backup
4. 🟡 **Arco Cirúrgico** — risco cirúrgico, mas tem 2 unidades (backup)
5. 🟡 **Mamógrafo** — rastreamento oncológico, sem backup
6. 🟡 **Raio-X Digital** — alta demanda, sem backup
7. 🟢 **Ultrassom** — impacto diluído entre 6 unidades


In [12]:
# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Cálculo dos Pesos AHP
#
# O AHP compara equipamentos par a par em cada critério.
# Aqui usamos uma versão simplificada (pesos diretos normalizados)
# adequada para simulação em TCC — o método completo com matriz
#
# Escala de pontuação:
#   5 = muito alta criticidade neste critério
#   4 = alta
#   3 = média
#   2 = baixa
#   1 = muito baixa
# ─────────────────────────────────────────────────────────────

# Pesos dos critérios (somam 1.0 = 100%)
pesos_criterios = {
    'risco_paciente':   0.40,   # Risco ao paciente em caso de falha
    'custo_equip':      0.25,   # Custo/valor de substituição
    'volume_exames':    0.20,   # Impacto operacional (volume de uso)
    'sem_backup':       0.15,   # Facilidade de backup (sem backup = mais crítico)
}

# Pontuação de cada equipamento em cada critério
# Quanto MAIOR a pontuação, MAIS crítico é o equipamento naquele critério
scores = {
    #                     risco  custo  volume sem_bkp
    'Tomografo':         [5,     5,     4,     5],  # sem backup, diagnóstico urgência
    'Ressonancia':       [5,     5,     3,     5],  # mais caro, sem backup
    'Mamografo':         [4,     3,     4,     5],  # rastreamento onco, sem backup
    'Arco_Cirurgico':    [5,     4,     2,     2],  # risco cirúrgico, tem backup
    'Ultrassom':         [3,     2,     5,     1],  # 6 unidades = muito backup
    'RaioX':             [3,     3,     5,     5],  # alta demanda, sem backup
    'Hemodinamica':      [5,     5,     2,     2],  # maior risco de vida, tem backup
}

# Aqui calculamos o peso final ponderado para cada equipamento!
criterios = list(pesos_criterios.keys())
pesos_list = list(pesos_criterios.values())

peso_ahp = {}
for eq, pontos in scores.items():
    # Multiplica cada pontuação pelo peso do critério e soma tudo, mitura aqui
    peso_ahp[eq] = sum(p * w for p, w in zip(pontos, pesos_list))

# Normaliza para que os pesos somem 1.0 (percentual de criticidade)
total = sum(peso_ahp.values())
peso_ahp_norm = {eq: round(v / total, 4) for eq, v in peso_ahp.items()}

# Painelzinho de amostra de resultados
print('=' * 55)
print('  RANKING DE CRITICIDADE — PESOS AHP NORMALIZADOS')
print('=' * 55)

ranking = sorted(peso_ahp_norm.items(), key=lambda x: x[1], reverse=True)
for i, (eq, peso) in enumerate(ranking, 1):
    nome = equipamentos[eq]['nome']
    barra = '█' * int(peso * 100)
    print(f'{i}. {nome:<35} {peso:.2%}  {barra}')

print('=' * 55)
print(f'Soma dos pesos: {sum(peso_ahp_norm.values()):.4f} (deve ser 1.0)')


  RANKING DE CRITICIDADE — PESOS AHP NORMALIZADOS
1. Tomógrafo Computadorizado           17.45%  █████████████████
2. Ressonância Magnética               16.73%  ████████████████
3. Hemodinâmica                        14.36%  ██████████████
4. Mamógrafo Digital                   14.18%  ██████████████
5. Arco Cirúrgico (Arco em C)          13.45%  █████████████
6. Raio-X Digital                      13.45%  █████████████
7. Aparelho de Ultrassom               10.36%  ██████████
Soma dos pesos: 0.9998 (deve ser 1.0)


---
## ⚙️ Bloco 4 — Configuração do Horizonte e Restrições

Configuramos aqui com oque temos dentro do nosso contexto de restrições, e técnicos !


In [13]:
# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Configuração do horizonte de planejamento
#
# Horizonte: ano de 2026 (12 meses)
# Cada manutenção preventiva é alocada em um mês do ano.
# O modelo decide em qual mês cada MP deve ocorrer.
# ─────────────────────────────────────────────────────────────

# Horizonte de planejamento: meses de 2026
MESES = list(range(1, 13))  # 1=Jan, 2=Fev, ... 12=Dez
NOMES_MESES = ['Jan','Fev','Mar','Abr','Mai','Jun',
               'Jul','Ago','Set','Out','Nov','Dez']

# ── Equipe de técnicos ────────────────────────────────────
# MODIFICADO AQUI
# O hospital possui 5 técnicos no total.
#
# De segunda a sexta:
#   - 2 técnicos ficam responsáveis pela rotina/atendimento hospitalar;
#   - 3 técnicos ficam dedicados às manutenções dos equipamentos.
#
# Aos sábados:
#   - 2 técnicos entram na escala de sobreaviso;
#   - 1 atende a rotina hospitalar;
#   - 1 fica dedicado às manutenções pendentes/atrasadas;
#   - os outros 3 técnicos ficam fora da escala naquele sábado.
#
# O rodízio é feito entre os 5 técnicos.
TECNICOS_DISPONIVEIS = 5
TECNICOS_MANUTENCAO_SEMANA = 3
TECNICOS_ROTINA_SEMANA = 2
TECNICOS_MANUTENCAO_SABADO = 1
TECNICOS_ROTINA_SABADO = 1

TECNICOS = [
    'Tecnico 1',
    'Tecnico 2',
    'Tecnico 3',
    'Tecnico 4',
    'Tecnico 5'
]

# MODIFICADO AQUI
# Escala de 5 sábados. O rodízio é feito entre os 5 técnicos.
# Em cada sábado:
# - 1 técnico fica na rotina;
# - 1 técnico fica na manutenção;
# - 3 técnicos ficam fora da escala.
#
# A função de manutenção passa de um técnico para outro
# ao longo dos sábados, e a função de rotina também é
# alternada para distribuir o sobreaviso.
ESCALA_SABADOS = [
    {'rotina': 'Tecnico 2', 'manutencao': 'Tecnico 1'},
    {'rotina': 'Tecnico 3', 'manutencao': 'Tecnico 2'},
    {'rotina': 'Tecnico 4', 'manutencao': 'Tecnico 3'},
    {'rotina': 'Tecnico 5', 'manutencao': 'Tecnico 4'},
    {'rotina': 'Tecnico 1', 'manutencao': 'Tecnico 5'}
]

# ── Horário da equipe terceirizada ────────────────────────
# As MPs externas continuam obedecendo ao horário da equipe
# terceirizada definido na simulação.
HORAS_UTEIS_MES = 176  # ~22 dias úteis × 8 horas

# MODIFICADO AQUI
# Conta quantos sábados existem em cada mês de 2026.
# Cada sábado acrescenta 8 horas de capacidade de manutenção,
# pois 1 técnico fica dedicado às manutenções nesse período.
SABADOS_POR_MES = {}

for mes in MESES:
    inicio_mes = pd.Timestamp(year=2026, month=mes, day=1)
    fim_mes = inicio_mes + pd.offsets.MonthEnd(1)

    dias_mes = pd.date_range(
        start=inicio_mes,
        end=fim_mes,
        freq='D'
    )

    SABADOS_POR_MES[mes] = sum(
        1 for dia in dias_mes
        if dia.weekday() == 5
    )

# Mostra a escala e a quantidade de sábados considerada.
print('Equipe total:', TECNICOS_DISPONIVEIS, 'técnicos')
print('Técnicos disponíveis para manutenção de segunda a sexta:',
      TECNICOS_MANUTENCAO_SEMANA)
print()
print('Escala de sobreaviso aos sábados:')
for i, escala in enumerate(ESCALA_SABADOS, start=1):
    print(
        f'Sábado {i}: rotina = {escala["rotina"]} | '
        f'manutenção = {escala["manutencao"]}'
    )

print()
print('Quantidade de sábados por mês:')
print(SABADOS_POR_MES)

# ── Gera as ordens de manutenção preventiva ───────────────
# Para cada equipamento e periodicidade, criamos as MP's do ano.
# Ex: Tomógrafo tem MP de 30 e 180 dias → MPS em jan, fev, mar...  e em jan e jul.

ordens_mp = []   # lista de todas as MPs a serem agendadas no ano

for eq_id, eq in equipamentos.items():
    for unidade in range(eq['qtd']):  # uma MP por unidade
        for periodo in eq['period_dias']:

            if periodo == 30:
                # Manutenção mensal — ocorre todo mês
                meses_alvo = MESES
            elif periodo == 90:
                # Trimestral — meses 1,4,7,10
                meses_alvo = [1, 4, 7, 10]
            elif periodo == 180:
                # Semestral — meses 1 e 7
                meses_alvo = [1, 7]
            else:
                meses_alvo = [1]

            for mes_venc in meses_alvo:

                # MODIFICAÇÃO
                # Antes: mes_limite = mes_venc (janela de 1 mês apenas).
                # Problema: com 200 MPs e capacidade limitada, o solver não
                # conseguia encaixar tudo — resultado: Infeasible.
                #
                # Agora: a MP pode ser feita a partir do mês ANTERIOR ao vencimento.
                # Para MPs de janeiro (mes_venc=1), o início é o próprio janeiro.
                # Para todas as outras, o início é mes_venc - 1.
                # O mes_limite continua sendo o mes_venc (prazo máximo preservado).
                #
                # Isso dá uma janela de 2 meses ao solver para distribuir
                # a carga entre meses consecutivos e resolver o Infeasible.
                mes_inicio = max(1, mes_venc - 1)   # MODIFICAÇÃO — janela real de alocação
                mes_limite = mes_venc                # prazo máximo (sem alteração)

                ordens_mp.append({
                    'id':           f'{eq_id}_U{unidade+1}_P{periodo}_{mes_venc:02d}',
                    'equip':        eq_id,
                    'unidade':      unidade + 1,
                    'periodo':      periodo,
                    'mes_venc':     mes_venc,
                    'mes_inicio':   mes_inicio,      # MODIFICAÇÃO — campo novo
                    'mes_limite':   mes_limite,
                    'tempo_h':      eq['tempo_mp_h'],
                    'tipo':         eq['tipo_mp'],
                    'tem_backup':   eq['tem_backup'],
                    'peso_ahp':     peso_ahp_norm[eq_id],
                })

print(f'✅ Total de ordens de MP geradas para 2026: {len(ordens_mp)}')
print(f'   Técnicos disponíveis: {TECNICOS_DISPONIVEIS}')
print(f'   Horas úteis por mês: {HORAS_UTEIS_MES}h')
print()

# Mostra um resumo das MPs por equipamento
df_ordens = pd.DataFrame(ordens_mp)
resumo = df_ordens.groupby('equip')['id'].count().reset_index()
resumo.columns = ['Equipamento', 'Total de MPs no ano']
resumo['Nome'] = resumo['Equipamento'].map({k: v['nome'] for k,v in equipamentos.items()})
print(resumo[['Nome','Total de MPs no ano']].to_string(index=False))


Equipe total: 5 técnicos
Técnicos disponíveis para manutenção de segunda a sexta: 3

Escala de sobreaviso aos sábados:
Sábado 1: rotina = Tecnico 2 | manutenção = Tecnico 1
Sábado 2: rotina = Tecnico 3 | manutenção = Tecnico 2
Sábado 3: rotina = Tecnico 4 | manutenção = Tecnico 3
Sábado 4: rotina = Tecnico 5 | manutenção = Tecnico 4
Sábado 5: rotina = Tecnico 1 | manutenção = Tecnico 5

Quantidade de sábados por mês:
{1: 5, 2: 4, 3: 4, 4: 4, 5: 5, 6: 4, 7: 4, 8: 5, 9: 4, 10: 5, 11: 4, 12: 4}
✅ Total de ordens de MP geradas para 2026: 200
   Técnicos disponíveis: 5
   Horas úteis por mês: 176h

                      Nome  Total de MPs no ano
Arco Cirúrgico (Arco em C)                   28
              Hemodinâmica                   32
         Mamógrafo Digital                   14
            Raio-X Digital                   14
     Ressonância Magnética                   14
 Tomógrafo Computadorizado                   14
     Aparelho de Ultrassom                   84


---
## 🧮 Bloco 5 — Busca em Grade Determinística

Nesta versão, o modelo PLIM continua responsável por encontrar a melhor programação **dentro de cada cenário**.

A diferença é que agora testamos sistematicamente diferentes configurações de planejamento por meio de uma **busca em grade (Grid Search)**.

### Estratégia

1. Definir os valores da grade antes da execução.
2. Resolver o PLIM para cada combinação.
3. Registrar se o cenário é factível.
4. Comparar as soluções factíveis por uma métrica comum.
5. Selecionar o melhor cenário para os blocos seguintes.

> **Importante:** a busca em grade não substitui o PLIM. Ela define os cenários que serão avaliados; o PLIM continua tomando as decisões `x[MP, mês]` dentro de cada cenário.


### 🔹 Bloco 5.1 — Simulação da demanda clínica

Primeiro mantemos a mesma simulação de demanda utilizada anteriormente, com semente fixa para garantir reprodutibilidade.


In [14]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.1 — Simulação da demanda clínica
# ─────────────────────────────────────────────────────────────

np.random.seed(42)

# Fator sazonal usado na simulação da demanda.
sazonalidade = [
    1.2, 1.1, 0.9, 0.9, 1.0, 1.3,
    1.2, 1.3, 1.0, 0.9, 0.9, 1.1
]

# Meses considerados de pico.
meses_pico = [
    i + 1 for i, s in enumerate(sazonalidade)
    if s > 1.1
]

# Demanda clínica mensal por equipamento.
demanda_clinica = {}

for eq_id, eq in equipamentos.items():

    base_exames_dia = 8

    demanda_clinica[eq_id] = {}

    for i, mes in enumerate(MESES):

        demanda_clinica[eq_id][mes] = int(
            base_exames_dia
            * 22
            * sazonalidade[i]
            * eq['qtd']
        )

print('Meses de pico:', meses_pico)
print()
print('Demanda clínica simulada criada com sucesso.')


Meses de pico: [1, 6, 7, 8]

Demanda clínica simulada criada com sucesso.


### 🔹 Bloco 5.2 — Definição da grade de busca

A grade abaixo define **quais cenários serão testados**.

A variável `JANELA_MESES` controla quantos meses antes do vencimento uma MP pode ser antecipada.

A variável `PENALIDADE_PICO` controla o peso dado à realização de MPs de equipamentos sem backup em meses de pico.

Com 3 valores de janela e 5 valores de penalidade, teremos:

**3 × 5 = 15 cenários.**


In [15]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.2 — Parâmetros da busca em grade
# ─────────────────────────────────────────────────────────────

GRADE_JANELA_MESES = [0, 1, 2]

GRADE_PENALIDADE_PICO = [
    0,
    250,
    500,
    1000,
    2000
]

# Penalidade usada SOMENTE para comparar os cenários
# com uma mesma métrica.
PENALIDADE_REFERENCIA_AVALIACAO = 1000

TOTAL_CENARIOS = (
    len(GRADE_JANELA_MESES)
    * len(GRADE_PENALIDADE_PICO)
)

print('=' * 65)
print('CONFIGURAÇÃO DA BUSCA EM GRADE')
print('=' * 65)
print('Janelas testadas:', GRADE_JANELA_MESES)
print('Penalidades testadas:', GRADE_PENALIDADE_PICO)
print('Total de cenários:', TOTAL_CENARIOS)
print('=' * 65)


CONFIGURAÇÃO DA BUSCA EM GRADE
Janelas testadas: [0, 1, 2]
Penalidades testadas: [0, 250, 500, 1000, 2000]
Total de cenários: 15


### 🔹 Bloco 5.3 — Função que resolve um cenário

Aqui construímos uma função separada para evitar um código enorme.

Cada chamada da função:

- cria um novo modelo PLIM;
- cria suas próprias variáveis de decisão;
- aplica as restrições;
- resolve o modelo;
- calcula os indicadores;
- devolve os resultados daquele cenário.


In [16]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.3 — Função de resolução de um cenário
# ─────────────────────────────────────────────────────────────

def resolver_cenario_grid(janela_meses, penalidade_mes_pico):

    # ── Cria um novo modelo para este cenário ─────────────
    modelo_cenario = pulp.LpProblem(
        f'Agendamento_MP_2026_J{janela_meses}_P{penalidade_mes_pico}',
        pulp.LpMinimize
    )

    # ── Variáveis de decisão ─────────────────────────────
    x_cenario = {}

    for mp in ordens_mp:

        for mes in MESES:

            # Corrigido usando argumentos nomeados (keyword arguments)
            # para garantir compatibilidade com as versões do PuLP.
            x_cenario[(mp['id'], mes)] = pulp.LpVariable(
                f"x_{mp['id']}_m{mes}",
                lowBound=0,
                upBound=1,
                cat=pulp.LpBinary
            )

    # ── Função objetivo ──────────────────────────────────
    custo_demanda = pulp.lpSum(
        mp['peso_ahp']
        * demanda_clinica[mp['equip']][mes]
        * x_cenario[(mp['id'], mes)]

        for mp in ordens_mp
        for mes in MESES
    )

    custo_mes_pico = pulp.lpSum(
        penalidade_mes_pico
        * mp['peso_ahp']
        * x_cenario[(mp['id'], mes)]

        for mp in ordens_mp
        for mes in meses_pico

        if not mp['tem_backup']
    )

    modelo_cenario += (
        custo_demanda + custo_mes_pico
    )

    # ── R1 — Cada MP ocorre exatamente uma vez ───────────
    for mp in ordens_mp:

        modelo_cenario += (
            pulp.lpSum(
                x_cenario[(mp['id'], mes)]
                for mes in MESES
            ) == 1,

            f'R1_unicidade_{mp["id"]}'
        )

    # ── R2 — Janela de antecipação ───────────────────────
    for mp in ordens_mp:

        mes_inicio = max(
            1,
            mp['mes_venc'] - janela_meses
        )

        mes_limite = mp['mes_limite']

        for mes in MESES:

            if (
                mes < mes_inicio
                or mes > mes_limite
            ):

                modelo_cenario += (
                    x_cenario[(mp['id'], mes)] == 0,

                    f'R2_janela_{mp["id"]}_m{mes}'
                )

    # ── R3 — Capacidade mensal em horas-técnico ──────────
    capacidade_cenario = {}

    for mes in MESES:

        horas_semana = (
            TECNICOS_MANUTENCAO_SEMANA
            * HORAS_UTEIS_MES
        )

        horas_sabado = (
            TECNICOS_MANUTENCAO_SABADO
            * 8
            * SABADOS_POR_MES[mes]
        )

        capacidade_cenario[mes] = (
            horas_semana
            + horas_sabado
        )

        modelo_cenario += (
            pulp.lpSum(
                mp['tempo_h']
                * x_cenario[(mp['id'], mes)]

                for mp in ordens_mp
            )
            <= capacidade_cenario[mes],

            f'R3_horas_manutencao_mes{mes}'
        )

    # ── Resolve o cenário ────────────────────────────────
    status = modelo_cenario.solve(
        pulp.PULP_CBC_CMD(msg=0)
    )

    status_nome = pulp.LpStatus[status]

    # Se o cenário não for ótimo/factível,
    # não tentamos acessar os valores das variáveis.
    if status_nome != 'Optimal':

        return {
            'janela_meses': janela_meses,
            'penalidade_pico': penalidade_mes_pico,
            'status': status_nome,
            'objetivo_modelo': None,
            'impacto_demanda': None,
            'mps_pico_sem_backup': None,
            'score_avaliacao': None,
            'total_horas': None,
            'modelo': modelo_cenario,
            'x': x_cenario,
            'capacidade_mensal': capacidade_cenario
        }

    # ── Indicadores da solução ───────────────────────────
    impacto_demanda = 0.0
    mps_pico_sem_backup = 0
    total_horas = 0.0

    for mp in ordens_mp:

        for mes in MESES:

            valor = pulp.value(
                x_cenario[(mp['id'], mes)]
            )

            if valor is not None and valor > 0.5:

                 impacto_demanda += (
                    mp['peso_ahp']
                    * demanda_clinica[mp['equip']][mes]
                )

                 total_horas += mp['tempo_h']

                 if (
                    mes in meses_pico
                    and not mp['tem_backup']
                 ):
                    mps_pico_sem_backup += 1

    # Métrica comum para comparar todos os cenários.
    score_avaliacao = (
        impacto_demanda
        + PENALIDADE_REFERENCIA_AVALIACAO
        * mps_pico_sem_backup
    )

    return {
        'janela_meses': janela_meses,
        'penalidade_pico': penalidade_mes_pico,
        'status': status_nome,
        'objetivo_modelo': pulp.value(modelo_cenario.objective),
        'impacto_demanda': impacto_demanda,
        'mps_pico_sem_backup': mps_pico_sem_backup,
        'score_avaliacao': score_avaliacao,
        'total_horas': total_horas,
        'modelo': modelo_cenario,
        'x': x_cenario,
        'capacidade_mensal': capacidade_cenario
    }

print('Função resolver_cenario_grid criada com sucesso.')

Função resolver_cenario_grid criada com sucesso.


### 🔹 Bloco 5.4 — Execução da busca em grade

Agora percorremos **todas as combinações** da grade.

Cada cenário é armazenado em uma tabela para que seja possível analisar posteriormente quais combinações foram factíveis e qual apresentou o melhor resultado.


In [17]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.4 — Execução da busca em grade
# ─────────────────────────────────────────────────────────────

resultados_grid = []
cenarios_factiveis = []

print('=' * 72)
print('🔎 INÍCIO DA BUSCA EM GRADE DETERMINÍSTICA')
print('=' * 72)

print('Janela de antecipação:', GRADE_JANELA_MESES)
print('Penalidades:', GRADE_PENALIDADE_PICO)
print('Total de combinações:', TOTAL_CENARIOS)
print()

for janela in GRADE_JANELA_MESES:

    for penalidade in GRADE_PENALIDADE_PICO:

        resultado = resolver_cenario_grid(
            janela_meses=janela,
            penalidade_mes_pico=penalidade
        )

        resultados_grid.append({
            'Janela (meses)': resultado['janela_meses'],
            'Penalidade pico': resultado['penalidade_pico'],
            'Status': resultado['status'],
            'Objetivo do modelo': resultado['objetivo_modelo'],
            'Impacto clínico': resultado['impacto_demanda'],
            'MPs pico sem backup': resultado['mps_pico_sem_backup'],
            'Score avaliação': resultado['score_avaliacao'],
            'Horas totais': resultado['total_horas']
        })

        if resultado['status'] == 'Optimal':
            cenarios_factiveis.append(resultado)

        print(
            f"Janela={janela} | "
            f"Penalidade={penalidade:>4} | "
            f"Status={resultado['status']}"
        )

# Tabela completa da busca.
df_grid = pd.DataFrame(
    resultados_grid
)

print()
print('=' * 72)
print('📊 RESULTADOS DA BUSCA EM GRADE')
print('=' * 72)

display(df_grid)


🔎 INÍCIO DA BUSCA EM GRADE DETERMINÍSTICA
Janela de antecipação: [0, 1, 2]
Penalidades: [0, 250, 500, 1000, 2000]
Total de combinações: 15



TypeError: LpVariable.__init__() got an unexpected keyword argument 'lowBound'

### 🔹 Bloco 5.5 — Seleção da melhor solução

Depois de testar toda a grade, selecionamos a melhor solução **entre os cenários factíveis**.

A comparação utiliza o mesmo `score_avaliacao` para todos os cenários. Assim, a penalidade utilizada internamente para resolver um cenário não favorece artificialmente esse próprio cenário na comparação final.


In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.5 — Seleção da melhor solução
# ─────────────────────────────────────────────────────────────

if not cenarios_factiveis:

    raise RuntimeError(
        'Nenhum cenário da busca em grade foi factível. '
        'Será necessário ampliar a janela ou revisar '
        'as restrições do modelo.'
    )

# Primeiro critério:
# menor score de avaliação.
#
# Critérios seguintes:
# menos MPs em meses de pico;
# menor janela de antecipação;
# menor penalidade utilizada no cenário.

melhor_cenario = min(
    cenarios_factiveis,

    key=lambda resultado: (
        resultado['score_avaliacao'],
        resultado['mps_pico_sem_backup'],
        resultado['janela_meses'],
        resultado['penalidade_pico']
    )
)

# Passamos a usar a solução vencedora
# nos blocos seguintes do notebook.
modelo = melhor_cenario['modelo']
x = melhor_cenario['x']
capacidade_mensal = melhor_cenario['capacidade_mensal']

PENALIDADE_MES_PICO = (
    melhor_cenario['penalidade_pico']
)

JANELA_MELHOR_CENARIO = (
    melhor_cenario['janela_meses']
)

print('=' * 72)
print('🏆 MELHOR CENÁRIO ENCONTRADO')
print('=' * 72)

print(
    f"Janela de antecipação: "
    f"{JANELA_MELHOR_CENARIO} mês(es)"
)

print(
    f"Penalidade de mês de pico: "
    f"{PENALIDADE_MES_PICO}"
)

print(
    f"Status: "
    f"{melhor_cenario['status']}"
)

print(
    f"Objetivo interno: "
    f"{melhor_cenario['objetivo_modelo']:.2f}"
)

print(
    f"Impacto clínico ponderado: "
    f"{melhor_cenario['impacto_demanda']:.2f}"
)

print(
    f"MPs em pico sem backup: "
    f"{melhor_cenario['mps_pico_sem_backup']}"
)

print(
    f"Score de avaliação: "
    f"{melhor_cenario['score_avaliacao']:.2f}"
)

print(
    f"Horas totais de manutenção: "
    f"{melhor_cenario['total_horas']:.1f} h"
)

print('=' * 72)


### 🔹 Bloco 5.6 — Diagnóstico da solução escolhida

Este bloco verifica rapidamente se a solução vencedora respeita a capacidade mensal de horas-técnico.

Ele também deixa registrado no notebook o total anual de horas de manutenção e a capacidade disponível.


In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOCO 5.6 — Diagnóstico da solução escolhida
# ─────────────────────────────────────────────────────────────

carga_teorica_total = sum(
    mp['tempo_h']
    for mp in ordens_mp
)

capacidade_anual_total = sum(
    capacidade_mensal[mes]
    for mes in MESES
)

print('=' * 65)
print('ANÁLISE DA CAPACIDADE DE HORAS')
print('=' * 65)

print(
    f'Horas totais de manutenção previstas: '
    f'{carga_teorica_total} h'
)

print(
    f'Capacidade anual da equipe: '
    f'{capacidade_anual_total} h'
)

print()

for mes in MESES:

    carga_mes = sum(
        mp['tempo_h']
        * (
            pulp.value(
                x[(mp['id'], mes)]
            ) or 0
        )

        for mp in ordens_mp
    )

    capacidade_mes = capacidade_mensal[mes]

    percentual = (
        carga_mes / capacidade_mes * 100
        if capacidade_mes > 0
        else 0
    )

    print(
        f'{NOMES_MESES[mes-1]}: '
        f'{carga_mes:.1f} h / '
        f'{capacidade_mes:.1f} h '
        f'({percentual:.1f}%)'
    )

print('=' * 65)
print('✅ Diagnóstico concluído.')


---
## 🗓️ Bloco 6 — Heatmap de Agendamento de Manutenções (2026)

Este é o heatmap que mostra, para cada equipamento e cada mês,
**quantas manutenções preventivas** foram alocadas pelo modelo.
O fundo mostra a demanda clínica (mais escuro = mais exames).


In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOCO 6 — Heatmap de agendamento
#
# Mostra: linhas = equipamentos | colunas = meses de 2026
# Cor de fundo = demanda clínica (mais azul = mais exames)
# Número na célula = quantidade de MPs alocadas pelo modelo
# ─────────────────────────────────────────────────────────────

# ── Coleta os resultados do modelo ───────────────────────
# Para cada equipamento e mês, conta quantas MPs foram alocadas
resultado = {}
nomes_equip = {k: v['nome'] for k, v in equipamentos.items()}

for eq_id in equipamentos:
    resultado[eq_id] = {}
    for mes in MESES:
        # Soma as MPs deste equipamento alocadas neste mês
        total_mp = sum(
            1 for mp in ordens_mp
            if mp['equip'] == eq_id
            and pulp.value(x[(mp['id'], mes)]) is not None
            and pulp.value(x[(mp['id'], mes)]) > 0.5
        )
        resultado[eq_id][mes] = total_mp

# ── Monta a matriz de demanda (fundo do heatmap) ─────────
# Linha = equipamento, Coluna = mês
equip_ids = list(equipamentos.keys())
nomes_linhas = [equipamentos[e]['nome'] for e in equip_ids]

matriz_demanda = np.array([
    [demanda_clinica[eq][mes] for mes in MESES]
    for eq in equip_ids
], dtype=float)

matriz_mp = np.array([
    [resultado[eq][mes] for mes in MESES]
    for eq in equip_ids
], dtype=float)

# ── Plota o heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))

# Fundo: demanda clínica (colormap azul — mais azul = mais exames)
im = ax.imshow(
    matriz_demanda,
    cmap='Blues',
    aspect='auto',
    alpha=0.7
)

# Adiciona os números (MPs alocadas) em cada célula
for i, eq in enumerate(equip_ids):
    for j, mes in enumerate(MESES):
        n_mp = int(matriz_mp[i, j])
        demanda = matriz_demanda[i, j]
        cor_texto = 'white' if demanda > matriz_demanda.max() * 0.6 else '#1a2535'

        if n_mp > 0:
            # Destaca células com manutenção
            ax.add_patch(plt.Rectangle(
                (j - 0.45, i - 0.45), 0.9, 0.9,
                fill=True, color='#e07d10', alpha=0.7, zorder=2
            ))
            ax.text(j, i, f'🔧\n{n_mp} MP',
                    ha='center', va='center', fontsize=8,
                    fontweight='bold', color='white', zorder=3)
        else:
            # Célula sem manutenção — mostra demanda
            ax.text(j, i, f'{int(demanda)}\nexames',
                    ha='center', va='center', fontsize=7.5,
                    color=cor_texto)

# ── Eixos e rótulos ───────────────────────────────────────
ax.set_xticks(range(12))
ax.set_xticklabels(NOMES_MESES, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(equip_ids)))
ax.set_yticklabels(nomes_linhas, fontsize=9)

# Destaca meses de pico (sazonalidade alta)
for j in range(12):
    if sazonalidade[j] > 1.1:
        ax.axvline(x=j, color='#dc2626', alpha=0.15, linewidth=8, zorder=0)

# Grade
ax.set_xticks(np.arange(-0.5, 12, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(equip_ids), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=1.5)

# Colorbar da demanda
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Volume de exames/mês', fontsize=9)

# Título e legenda
ax.set_title(
    'Heatmap de Agendamento de Manutenção Preventiva × Demanda Clínica — 2026\n'
    'Modelo de Pesquisa Operacional (PLIM) | Engenharia Hospitalar',
    fontsize=12, fontweight='bold', pad=15
)

legenda = [
    mpatches.Patch(color='#e07d10', label='🔧 MP alocada pelo modelo'),
    mpatches.Patch(color='#dc2626', alpha=0.3, label='📈 Mês de alta demanda clínica'),
    mpatches.Patch(color='#2563eb', alpha=0.5, label='🔵 Fundo = volume de exames'),
]
ax.legend(handles=legenda, loc='upper right', bbox_to_anchor=(1.28, 1), fontsize=8)

plt.tight_layout()
plt.savefig('heatmap_manutencao_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap gerado e salvo como heatmap_manutencao_2026.png')


---
## 📊 Bloco 7 — Tabela de Resultados e Indicadores

Aqui vemos o cronograma completo das manutenções e os indicadores
que você pode usar para comparar ANTES × DEPOIS da implantação do sistema.


In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOCO 7 — Tabela de resultados e indicadores do modelo
# ─────────────────────────────────────────────────────────────

# ── Monta a tabela de cronograma ─────────────────────────
linhas = []
for mp in ordens_mp:
    for mes in MESES:
        val = pulp.value(x[(mp['id'], mes)])
        if val is not None and val > 0.5:
            linhas.append({
                'Equipamento':      equipamentos[mp['equip']]['nome'],
                'Unidade':          mp['unidade'],
                'Tipo de MP':       f"{mp['periodo']} dias",
                'Mês alocado':      NOMES_MESES[mes - 1],
                'Mês limite':       NOMES_MESES[mp['mes_limite'] - 1],
                'Duração (h)':      mp['tempo_h'],
                'Tipo execução':    mp['tipo'].capitalize(),
                'Peso AHP':         f"{mp['peso_ahp']:.2%}",
                'Backup?':          '✅ Sim' if mp['tem_backup'] else '❌ Não',
            })

df_resultado = pd.DataFrame(linhas)
df_resultado = df_resultado.sort_values(['Mês alocado', 'Equipamento'])

print('=' * 65)
print('   CRONOGRAMA ÓTIMO DE MANUTENÇÕES PREVENTIVAS — 2026')
print('=' * 65)
print(df_resultado.to_string(index=False))

# ── Indicadores do modelo ─────────────────────────────────
print()
print('=' * 50)
print('   INDICADORES DO MODELO')
print('=' * 50)

total_mps = len(df_resultado)
total_horas = df_resultado['Duração (h)'].sum()
mps_sem_backup = df_resultado[df_resultado['Backup?'] == '❌ Não'].shape[0]
mps_pico = df_resultado[
    df_resultado['Mês alocado'].isin([NOMES_MESES[m-1] for m in meses_pico])
].shape[0]

print(f'Total de MPs alocadas no ano:      {total_mps}')
print(f'Total de horas de manutenção:      {total_horas}h')
print(f'MPs sem equipamento backup:        {mps_sem_backup}')
print(f'MPs em meses de alta demanda:      {mps_pico}  ← ideal = 0')
print(f'Técnicos no quadro total:            {TECNICOS_DISPONIVEIS}')
print(f'Técnicos em manutenção (seg-sex):    {TECNICOS_MANUTENCAO_SEMANA}')
print(f'Técnicos em manutenção (sábado):     {TECNICOS_MANUTENCAO_SABADO}')
print()
print('📌 Meses de alta demanda clínica (sazonalidade > 1.1):')
print('  ', [NOMES_MESES[m-1] for m in meses_pico])

# ── Gráfico de carga por mês ──────────────────────────────
carga_mes = df_resultado.groupby('Mês alocado')['Duração (h)'].sum()
# Reordena pelos meses
carga_mes = carga_mes.reindex([NOMES_MESES[i] for i in range(12)], fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico 1: Horas de MP por mês
bars = ax1.bar(carga_mes.index, carga_mes.values, color='#0A6E63', alpha=0.8)
ax1.plot(
    [NOMES_MESES[i] for i in range(12)],
    [capacidade_mensal[i+1] for i in range(12)],
    linestyle='--',
    label='Capacidade de manutenção do mês',
    linewidth=1.5
)
ax1.set_title('Carga de Manutenção por Mês (horas)', fontweight='bold')
ax1.set_ylabel('Horas de MP')
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=45)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{int(bar.get_height())}h', ha='center', va='bottom', fontsize=8)

# Gráfico 2: Número de MPs por equipamento
mp_por_equip = df_resultado['Equipamento'].value_counts()
ax2.barh(mp_por_equip.index, mp_por_equip.values, color='#e07d10', alpha=0.8)
ax2.set_title('Quantidade de MPs por Equipamento', fontweight='bold')
ax2.set_xlabel('Número de MPs no ano')
for i, v in enumerate(mp_por_equip.values):
    ax2.text(v + 0.1, i, str(v), va='center', fontsize=9)

plt.suptitle('Indicadores do Modelo PLIM — Agendamento Hospitalar 2026',
             fontweight='bold', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('indicadores_mp_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráficos salvos como indicadores_mp_2026.png')


---
## 📚 Referências utilizadas no código

| Item | Referência | Link |
|------|-----------|------|
| Tempo MP — TC e RM | VX Med — Manutenção de Equipamentos Radiológicos | https://vx.med.br/manutencao-de-equipamentos-radiologicos/ |
| Tempo MP — Ultrassom | POP-STEC-079 HU-UFGD / Portaria 094/2025 | https://www.gov.br/hubrasil/.../pop-stec-079... |
| Periodicidade — imagem | Portaria SVS/MS nº 453/1998 (ANVISA) | https://www.legisweb.com.br/legislacao/?id=181745 |
| Periodicidade — hemodinâmica | RDC 611/2022 (ANVISA) | https://www.sapralandauer.com.br/wp-content/uploads/2022/05/RESOLUÇÃO-RDC-611-2022.pdf |
| Tempo exame — TC | Hospital São José (15–30 min) | https://www.hsjose.com.br/exames/centro-diagnosticos/tomografia |
| Tempo exame — RM | Blume Diagnósticos | https://blumediagnosticos.com.br/exames-de-imagem-... |
| Periodicidade geral | RDC 16/2013 (ANVISA) | https://bvsms.saude.gov.br/bvs/saudelegis/anvisa/2013/rdc0016_28_03_2013.pdf |
| Método AHP | Masmoudi et al. (2016) — Journal of Clinical Engineering | DOI: 10.1097/JCE.0000000000000089 |
